# KVQuant Implementation -- 3-bit only

One of four split-out notebooks (2-bit / 3-bit / 4-bit / full-precision
baseline), each running independently so they can execute in parallel
Colab sessions instead of one long combined run. This notebook is
identical in methodology to `KVQuant_Implementation_v3.ipynb`, scoped to
just the 3-bit quantized model -- no other bit-width, no baseline model,
loaded or evaluated here (the baseline lives in its own dedicated
notebook, `KVQuant_Baseline_Implementation.ipynb`, so it only needs to run
once rather than being redundantly repeated in all three quantized
notebooks).

Every fix already built into v3 is preserved as-is: real hook-based KV
cache memory measurement (via `register_outlier_trackers`, using the
actual `get_outliers`/`get_outliers_dynamic` functions and the model's own
real calibrated thresholds -- not an assumed fixed sparsity fraction),
teacher-forced step-by-step TTFT/TBT timing for WikiText-103, real
`generate()`-based GSM8K timing with TBT excluding TTFT, H2O-matching
sampling (full-text-then-chunk for WikiText-103, sequential first
`N_QA_SAMPLES` questions for GSM8K, no shuffling), and
download-retry-with-cache-clear robustness for every network call.

Run cells top to bottom. Needs a GPU runtime.

## Setup

In [ ]:
# Block 1 - Environment setup
# Run once per fresh runtime. Package versions are pinned so environment
# differences are never a confound between compression methods.

import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

from google.colab import drive
drive.mount("/content/drive")

!python -m pip install -q --no-deps \
  "transformers==4.43.4" \
  "accelerate==0.33.0" \
  "tokenizers==0.19.1" \
  "huggingface_hub==0.36.2" \
  sentencepiece \
  einops

!python -m pip install -q \
  "datasets==2.14.5" \
  tqdm \
  matplotlib

!python -m pip install -q --no-deps --force-reinstall "huggingface_hub==0.36.2"

import os

os.environ["HF_HUB_DISABLE_XET"] = "1"

try:
    from google.colab import userdata
    _hf_token = userdata.get("HF_TOKEN")
except Exception:
    _hf_token = os.environ.get("HF_TOKEN")

if _hf_token:
    from huggingface_hub import login
    login(token=_hf_token)
    print("Logged in to HuggingFace")
else:
    print("No HF_TOKEN found -- skipping login (fine for public models like TinyLlama)")

print("Block 1 finished. Now run Block 2.")

In [ ]:
# Block 2 - Imports, GPU check

import gc
import math
import re
import time
import random
import pickle
import sys

import numpy as np
import torch
import torch.nn as nn
import pandas as pd

import datasets
import transformers
import huggingface_hub
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, StoppingCriteria, StoppingCriteriaList

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("datasets:", datasets.__version__)
print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO CUDA")

HAS_CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if HAS_CUDA else "cpu")
MODEL_DTYPE = torch.float16 if HAS_CUDA else torch.float32


def clear_hf_dataset_cache(*dataset_names):
    """Removes cached files for the given HF dataset repo name(s) (e.g.
    "wikitext", "gsm8k") from both the datasets cache and the hub cache.
    Used as an on_retry hook: a download that breaks partway through can
    leave a corrupted partial file that every subsequent retry just
    resumes (and re-breaks at the same point) instead of truly restarting
    -- clearing the cache forces a genuinely fresh download."""
    home = os.path.expanduser("~")
    for name in dataset_names:
        for base in [
            os.path.join(home, ".cache", "huggingface", "datasets", name),
            os.path.join(home, ".cache", "huggingface", "hub", f"datasets--{name}"),
        ]:
            shutil.rmtree(base, ignore_errors=True)


def robust_call(fn, *args, max_retries=5, backoff_sec=5, desc="operation", on_retry=None, **kwargs):
    """Retries fn(*args, **kwargs) on any exception, up to max_retries times,
    waiting backoff_sec between attempts -- guards dataset downloads against
    transient network failures (e.g. IncompleteRead/ChunkedEncodingError)
    rather than letting one flaky connection kill the whole notebook run.
    If on_retry is given, it's called (no args) after each failure, before
    the next attempt -- e.g. clear_hf_dataset_cache, so a retry that hit a
    stuck/corrupted partial download actually starts fresh instead of
    resuming (and re-breaking at) the same point every time."""
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            last_err = e
            _msg = f"  {desc}: attempt {attempt}/{max_retries} failed ({e!r})"
            if attempt < max_retries:
                _msg += f", retrying in {backoff_sec}s..."
            print(_msg)
            if attempt < max_retries:
                if on_retry is not None:
                    on_retry()
                time.sleep(backoff_sec)
    raise last_err


if not HAS_CUDA:
    print("WARNING: No GPU detected. This will be very slow.")

# NOTE: clear_hf_dataset_cache/robust_call are defined here (not in Helper
# Functions, below) because the Fisher calibration cell later in this same
# Setup section calls robust_call for its wikitext-2 prefetch -- they need
# to exist before that point. sync_if_cuda/clear_memory have no such early
# dependency and live in Helper Functions with the rest of the genuinely
# cross-dataset inference machinery.

In [ ]:
# Block 3 - Experiment settings.
# Kept identical in spirit to H2O_Implementation_v3.ipynb wherever the same
# quantity applies (C, SHARED_SEED, GSM8K prompt, N_EVAL_SAMPLES,
# N_QA_SAMPLES,
# GSM8K_MAX_NEW_TOKENS) so the two methods' evaluations are only different
# where the compression method itself makes them different. This notebook
# only ever builds/evaluates ONE bit-width (3-bit) -- 2-bit, 4-bit, and the
# full-precision baseline each live in their own separate notebook.

LOCAL_MODEL_PATH = "/content/tinyllama-1.1b"
HF_MODEL_ID = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
MODEL_ID = LOCAL_MODEL_PATH if __import__("os").path.exists(LOCAL_MODEL_PATH) else HF_MODEL_ID

SHARED_SEED = 0
C = 2048                    # chunk length for WikiText-103 = model's max context
N_EVAL_SAMPLES = 64          # caps WikiText-103 source prompts (chunk count)
N_QA_SAMPLES = 256           # GSM8K and ARC-Challenge question count
GSM8K_MAX_NEW_TOKENS = 256
METHOD_NAME = "kvquant_3bit"

ABITS = 3                   # this notebook's single bit-width
SPARSITY_THRESHOLD = 0.99   # keep the extreme ~1% of values in full precision
FISHER_OUTPUT_DIR = "/content/fisher-tinyllama-1.1b-c2048"
QUANTIZER_PATH = "/content/quantizers_tinyllama-1.1b_v3_c2048_abits3.pickle"

random.seed(SHARED_SEED)
torch.manual_seed(SHARED_SEED)

GSM8K_FEWSHOT_PREFIX = (
    "You are solving grade-school math word problems.\n"
    "Show the calculation briefly, then end with exactly this format:\n"
    "#### <final number>\n\n"
    "Question: Mia has 3 marbles and buys 4 more. How many marbles does she have?\n"
    "Answer: Mia has 3 + 4 = 7 marbles.\n"
    "#### 7\n\n"
    "Question: A box has 6 pencils. Sam buys 5 boxes. How many pencils does Sam buy?\n"
    "Answer: Sam buys 6 * 5 = 30 pencils.\n"
    "#### 30\n"
)

print("Model:", MODEL_ID)
print("Method:", METHOD_NAME)
print("Chunk length C:", C)
print("Bit-width:", ABITS, "| sparsity threshold:", SPARSITY_THRESHOLD)
print("N_EVAL_SAMPLES (WikiText-103 chunks):", N_EVAL_SAMPLES)
print("N_QA_SAMPLES (GSM8K/ARC-Challenge questions):", N_QA_SAMPLES)
print("GSM8K max new tokens:", GSM8K_MAX_NEW_TOKENS)

In [ ]:
# Repo setup 1/3 - Clone repo (always fresh, so the rotary_emb patch below
# always applies to pristine upstream source rather than a possibly-
# already-patched copy left over from an earlier run in this session).
import os, shutil


def check_path(path, label):
    if not os.path.exists(path):
        raise FileNotFoundError(f"ERROR: {label} not found at {path}")
    print(f"OK: {label} found at {path}")


if os.path.exists("/content/KVCacheCompression"):
    shutil.rmtree("/content/KVCacheCompression")
    print("Removed existing repo copy for a clean re-clone")

!git clone --recurse-submodules https://github.com/yoshikodes/KVCacheCompression.git

assert os.path.exists("/content/KVCacheCompression/KVQuant/gradients/run-fisher.py"), \
    "ERROR: run-fisher.py not found! Clone may have failed."
assert os.path.exists("/content/KVCacheCompression/KVQuant/quant/llama_simquant.py"), \
    "ERROR: llama_simquant.py not found! Clone may have failed."
print("Repo verified OK")

In [ ]:
# Repo setup 2/3 - Install gradients (Fisher calibration) dependencies
check_path("/content/KVCacheCompression/KVQuant/gradients", "gradients dir")

%cd /content/KVCacheCompression/KVQuant/gradients
!pip install -e . -q
!pip install datasets==2.14.5 sentencepiece==0.1.99 accelerate -q -U
!pip install peft==0.6.0 -q
print("Gradients deps installed")

In [ ]:
# Patch - llama_simquant.py: move the shared rotary embedding module to the
# target device before calibration. transformers==4.43.4 computes rotary
# embeddings ONCE at the top-level LlamaModel, not per-layer -- the upstream
# script's custom per-layer CPU->GPU transfer (written for an older
# architecture) never moves this shared module, so it stays on CPU while
# everything it's multiplied against is on GPU. Only needed for the
# calibration path here (this notebook's own evaluation, below, uses a
# normally-loaded model and never calls llama_simquant.py's llama_eval).
filepath = "/content/KVCacheCompression/KVQuant/quant/llama_simquant.py"
with open(filepath, "r") as f:
    _content = f.read()

if "model.model.rotary_emb = model.model.rotary_emb.to(DEV)" not in _content:
    _m = re.search(r"^([ \t]*)quantizers = llama_calibration\(", _content, re.MULTILINE)
    assert _m, "ERROR: could not locate the llama_calibration(...) call to patch"
    _indent = _m.group(1)
    _insertion = (
        f"{_indent}if hasattr(model, 'model') and hasattr(model.model, 'rotary_emb'):\n"
        f"{_indent}    model.model.rotary_emb = model.model.rotary_emb.to(DEV)\n"
    )
    _content = _content[:_m.start()] + _insertion + _content[_m.start():]
    with open(filepath, "w") as f:
        f.write(_content)
    print("Patched llama_simquant.py: rotary_emb moved to device before calibration")
else:
    print("rotary_emb device patch already applied, skipping")

In [ ]:
# Patch - llama_simquant.py: get_model() hardcodes
# use_flash_attention_2=True, which crashes with "flash_attn seems to be not
# installed" since this project never installs the flash-attn package (it's
# optional elsewhere and skipped here entirely). Replace it with a
# GPU-capability-aware attn_implementation choice that also respects a
# FORCE_ATTN_IMPL env var, matching KVQuant_Implementation_v2.ipynb's Patch
# 2.4. Without this, both the Quantize step and this notebook's own
# in-process model loading would need it -- but the in-process loader
# already builds its own model_kwargs directly (no use_flash_attention_2
# anywhere in this notebook's own code), so only llama_simquant.py's
# internal get_model() needs patching, for the Quantize step.
filepath = "/content/KVCacheCompression/KVQuant/quant/llama_simquant.py"
with open(filepath, "r") as f:
    _content = f.read()

_old_load = "model = AutoModelForCausalLM.from_pretrained(model, config=config, trust_remote_code=True, use_flash_attention_2=True, torch_dtype=torch.half)"
_new_load = (
    "import warnings as _warnings\n"
    "    import os as _attn_os\n"
    "    import torch as _torch_gpu_check\n"
    "    _cap = _torch_gpu_check.cuda.get_device_capability(0) if _torch_gpu_check.cuda.is_available() else (0, 0)\n"
    "    _attn_impl = \"flash_attention_2\" if _cap[0] >= 8 else \"sdpa\"\n"
    "    _force_attn = _attn_os.environ.get('FORCE_ATTN_IMPL')\n"
    "    if _force_attn:\n"
    "        _attn_impl = _force_attn\n"
    "    print(f'GPU compute capability: {_cap} -> using attn_implementation={_attn_impl}' + (' (forced)' if _force_attn else ''))\n"
    "    with _warnings.catch_warnings():\n"
    "        _warnings.filterwarnings(\"ignore\", message=\".*Flash Attention 2.0 with a model not initialized on GPU.*\")\n"
    "        model = AutoModelForCausalLM.from_pretrained(model, config=config, trust_remote_code=True, attn_implementation=_attn_impl, torch_dtype=torch.half)"
)
if _old_load in _content:
    _content = _content.replace(_old_load, _new_load)
    with open(filepath, "w") as f:
        f.write(_content)
    print("llama_simquant.py attn_implementation patch applied (GPU-capability-aware)")
elif "GPU compute capability" in _content:
    print("attn_implementation patch already applied, skipping")
else:
    print("WARNING: could not find the expected model-loading line to patch.")
print("Has GPU-capability check:", "GPU compute capability" in open(filepath).read())

In [ ]:
# Patch - simquant_module_quantizer.py: QuantLinearSim.__init__ builds
# self.outlier_threshold_upper/lower via torch.tensor(already_a_tensor),
# which PyTorch flags with a UserWarning ("recommended to use
# sourceTensor.detach().clone()..."). Purely cosmetic -- quantizer[0]/[1]
# are already tensors (SimQuant.quantize() builds them that way before
# pickling), so this produces a numerically identical copy either way --
# but silenced here since it's a one-line-per-side change and this file
# gets read fresh from a clean clone every run.
filepath = "/content/KVCacheCompression/KVQuant/quant/kvquant/simquant_module_quantizer.py"
with open(filepath, "r") as f:
    _content = f.read()

_old_thresh = (
    "        self.outlier_threshold_upper = torch.tensor(quantizer[0]).cuda().flatten().half()\n"
    "        self.outlier_threshold_lower = torch.tensor(quantizer[1]).cuda().flatten().half()"
)
_new_thresh = (
    "        self.outlier_threshold_upper = quantizer[0].clone().detach().cuda().flatten().half()\n"
    "        self.outlier_threshold_lower = quantizer[1].clone().detach().cuda().flatten().half()"
)
if _old_thresh in _content:
    _content = _content.replace(_old_thresh, _new_thresh)
    with open(filepath, "w") as f:
        f.write(_content)
    print("Patched simquant_module_quantizer.py: outlier thresholds built via .clone().detach() instead of torch.tensor(tensor)")
elif "quantizer[0].clone().detach()" in _content:
    print("simquant_module_quantizer.py outlier-threshold patch already applied, skipping")
else:
    print("WARNING: could not find the expected outlier-threshold lines to patch (cosmetic only -- safe to ignore if this shows up).")

In [ ]:
# Fisher calibration (gradients) -- seqlen/maxseqlen MUST match the Quantize
# step's seqlen below (C = 2048), or Fisher's saved tensors won't align with
# Quantize's activation tensor shapes elementwise.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Pre-fetch wikitext-2 (with retries) in this notebook's own Python process
# before launching the Fisher subprocess -- run-fisher.py internally calls
# load_dataset('wikitext', 'wikitext-2-raw-v1', ...) itself, and a flaky
# connection there (IncompleteRead/ChunkedEncodingError) can crash the
# whole subprocess uncaught, leaving FISHER_OUTPUT_DIR never created.
print("Pre-fetching wikitext-2 (train+test) with retries...")
robust_call(
    load_dataset, "wikitext", "wikitext-2-raw-v1", split="train",
    desc="wikitext-2 train prefetch", on_retry=lambda: clear_hf_dataset_cache("wikitext"),
)
robust_call(
    load_dataset, "wikitext", "wikitext-2-raw-v1", split="test",
    desc="wikitext-2 test prefetch", on_retry=lambda: clear_hf_dataset_cache("wikitext"),
)
print("wikitext-2 cached.")


# Pre-fetch the TinyLlama model weights (with retries), using
# huggingface_hub.snapshot_download directly -- NOT transformers'
# AutoModelForCausalLM. By this point in the notebook, Cell 5's gradients
# install has replaced the real `transformers` package on disk with its
# own vendored fork (intentional -- run-fisher.py's subprocess needs it),
# but this kernel already imported the REAL transformers back in Cell 2.
# Calling AutoModelForCausalLM.from_pretrained() here would use those
# stale cached (real-transformers) references against fork files now on
# disk -- a mismatch that throws ModuleNotFoundError. snapshot_download
# only touches huggingface_hub, which isn't affected by any of this, so
# it's safe to call in-process at this point in the notebook.
from huggingface_hub import snapshot_download

print(f"Pre-fetching {MODEL_ID} weights with retries (this can take a few minutes)...")
robust_call(
    snapshot_download, MODEL_ID,
    desc=f"{MODEL_ID} weights prefetch",
    max_retries=5, backoff_sec=10,
    on_retry=lambda: clear_hf_dataset_cache(),  # no-op placeholder; see note below
)
print(f"{MODEL_ID} weights cached.")

%cd /content/KVCacheCompression/KVQuant/gradients

!CUDA_VISIBLE_DEVICES=0 PYTHONPATH=/content/KVCacheCompression/KVQuant/gradients/src python run-fisher.py --model_name_or_path {MODEL_ID} --output_dir {FISHER_OUTPUT_DIR} --dataset wikitext2 --seqlen {C} --maxseqlen {C} --num_examples 8

check_path(FISHER_OUTPUT_DIR, "Fisher output")
print("Fisher done for TinyLlama-1.1B at seqlen", C)

In [ ]:
# Repo setup 3/3 - Install quant (eval-support) dependencies. Moved to run
# AFTER Fisher calibration (matching KVQuant_Implementation_v2.ipynb's
# proven cell ordering) since gradients needs tokenizers<0.19 while this
# pins tokenizers==0.19.1 -- installing this first would break Fisher.
check_path("/content/KVCacheCompression/KVQuant/quant", "quant dir")

%cd /content/KVCacheCompression/KVQuant/quant
!pip install -e . -q --no-deps

!pip install -q --no-deps \
  "transformers==4.43.4" \
  "accelerate==0.33.0" \
  "tokenizers==0.19.1" \
  "huggingface_hub==0.36.2"

print("Quant deps installed, package versions pinned to match every other method's notebook")

In [ ]:
# Quantize (nuq3, 1% outliers) -- builds the single quantizer pickle this
# notebook's own in-process eval loop will load later. Note this only runs
# llama_calibration internally (the --quantize branch), never llama_eval --
# so the rotary_emb patch above is the only one needed.
sys.path = [p for p in sys.path if "gradients" not in p]

check_path(FISHER_OUTPUT_DIR, "Fisher output for TinyLlama-1.1B")

%cd /content/KVCacheCompression/KVQuant/quant

!CUDA_VISIBLE_DEVICES=0 PYTHONPATH=/content/KVCacheCompression/KVQuant/quant FORCE_ATTN_IMPL=sdpa python llama_simquant.py {MODEL_ID} --abits {ABITS} --nsamples 8 --seqlen {C} --nuq --fisher {FISHER_OUTPUT_DIR} --quantize --include_sparse --sparsity-threshold {SPARSITY_THRESHOLD} --quantizer-path {QUANTIZER_PATH}

check_path(QUANTIZER_PATH, "Quantizer pickle for TinyLlama-1.1B")
print("Quantization done for TinyLlama-1.1B at seqlen", C, "bit-width", ABITS)

In [ ]:
# Block - Load tokenizer + the single 3-bit quantized model (via
# make_quant_sim applied in-process, matching llama_simquant.py's own
# main-flow layer-name matching: Keys quantized per-channel pre-RoPE,
# Values quantized per-token). No full-precision baseline is loaded here
# -- that model lives entirely in KVQuant_Baseline_Implementation.ipynb.
#
# Also registers forward hooks on every quantized k_proj/v_proj
# (QuantLinearSim) submodule that recompute the REAL outlier mask the
# model actually uses internally -- via the exact same get_outliers /
# get_outliers_dynamic functions and the SAME stored threshold tensors the
# layer itself uses -- so memory can later be measured from the real,
# data-dependent outlier counts rather than an assumed fixed fraction.
# These hooks only OBSERVE (read the layer's real input, recompute its
# projection output independently); they never modify what the model
# actually computes or caches.
#
# The hooks stay attached for the model's whole lifetime (forward hooks
# fire on EVERY call to that module, including the TIMED step-by-step loop
# and the TIMED generate() call, not just the dedicated untimed memory
# passes) -- so _tracker_enabled gates the hook's actual work: it's a
# no-op unless explicitly turned on around the untimed measurement passes
# (see measure_chunk_kv_memory / generate_gsm8k_kvquant below). Without
# this gate, every timed forward call would redundantly pay the hook's
# extra x @ weight recompute plus the full outlier-mask computation,
# inflating TTFT/TBT/latency with overhead the full-precision baseline
# never pays (memory/perplexity/accuracy are unaffected either way, since
# the hook never touches the model's actual output, and
# reset_outlier_tracker already discards any stray accumulation before
# each real measurement -- this gate only removes wasted work, not a
# correctness bug in the reported numbers themselves).
%cd /content/KVCacheCompression/KVQuant/quant
sys.path.insert(0, "/content/KVCacheCompression/KVQuant/quant")

from kvquant.simquant_module_quantizer import make_quant_sim, get_outliers, get_outliers_dynamic, QuantLinearSim

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=False, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

_model_kwargs = {
    "torch_dtype": MODEL_DTYPE,
    "low_cpu_mem_usage": True,
    "attn_implementation": "sdpa",
    "trust_remote_code": True,
}
if HAS_CUDA:
    _model_kwargs["device_map"] = {"": 0}

print(f"Loading model to be quantized at {ABITS}-bit...")
model_q = AutoModelForCausalLM.from_pretrained(MODEL_ID, **_model_kwargs)
if not HAS_CUDA:
    model_q = model_q.to(DEVICE)
model_q.eval()
model_q.config.use_cache = True

device = next(model_q.parameters()).device

with open(QUANTIZER_PATH, "rb") as f:
    quantizers = pickle.load(f)

PERCHANNEL_MATCH = ["k_proj"]
PERTOKEN_MATCH = ["v_proj"]

perchannelquant = {}
pertokenquant = {}
for _k in quantizers.keys():
    for _p in PERCHANNEL_MATCH:
        if _p in _k:
            perchannelquant[_k] = quantizers[_k]
    for _p in PERTOKEN_MATCH:
        if _p in _k:
            pertokenquant[_k] = quantizers[_k]

print(f"Quantizer covers {len(perchannelquant)} per-channel (K) layers, {len(pertokenquant)} per-token (V) layers")

make_quant_sim(
    model_q, perchannelquant, ABITS,
    perchannel=True, include_sparse=True, sparsity_threshold=SPARSITY_THRESHOLD,
    dynamicquantization=False, nuq=True, nf_nuq=False, norm=False,
    cap_outliers=-1, first_few_fp16=-1, clamp=False,
)
make_quant_sim(
    model_q, pertokenquant, ABITS,
    perchannel=False, include_sparse=True, sparsity_threshold=SPARSITY_THRESHOLD,
    dynamicquantization=True, nuq=True, nf_nuq=False, norm=False,
    cap_outliers=-1, first_few_fp16=-1, clamp=False,
)

_tracker_enabled = [False]  # gate for the hooks below -- see comment above


def register_outlier_trackers(model_obj):
    """Attaches a forward hook to every QuantLinearSim submodule that
    independently recomputes that layer's projection output (x @ weight +
    bias -- identical to QuantLinearSim.forward()'s own math, computed
    BEFORE any RoPE/reshaping) and re-derives the REAL outlier mask via the
    same get_outliers/get_outliers_dynamic functions and stored threshold
    tensors the layer itself uses. Returns (tracker, handles) -- tracker
    accumulates real n_outliers/n_total per layer; call
    reset_outlier_tracker(tracker) before each measurement pass. The hook
    only does its work while _tracker_enabled[0] is True (see module-level
    note above) -- callers must toggle that flag on/off around the
    specific untimed forward pass they want measured.
    """
    tracker = {}
    handles = []

    for _name, _module in model_obj.named_modules():
        if not isinstance(_module, QuantLinearSim):
            continue
        tracker[_name] = {"n_outliers": 0, "n_total": 0}

        def _make_hook(mod, key):
            def _hook(_mod, inputs, _output):
                if not _tracker_enabled[0]:
                    return
                x = inputs[0].reshape(-1, inputs[0].shape[-1]).half()
                weight = mod.weight.to(x.device)
                y = x @ weight
                if mod.bias is not None:
                    y = y + mod.bias.to(x.device)
                y = y.float()

                if mod.include_sparse:
                    if mod.dynamicquantization:
                        mask = get_outliers_dynamic(
                            y, channel=mod.ochannel, thresh=mod.sparsity_threshold,
                            first_few_fp16=mod.first_few_fp16,
                        )
                    else:
                        mask = get_outliers(
                            y, channel=mod.ochannel,
                            outlier_threshold_upper=mod.outlier_threshold_upper,
                            outlier_threshold_lower=mod.outlier_threshold_lower,
                            cap_outliers=mod.cap_outliers,
                            first_few_fp16=mod.first_few_fp16,
                        )
                    tracker[key]["n_outliers"] += int(mask.sum().item())
                tracker[key]["n_total"] += int(y.numel())
            return _hook

        handles.append(_module.register_forward_hook(_make_hook(_module, _name)))

    return tracker, handles


def reset_outlier_tracker(tracker):
    for _counts in tracker.values():
        _counts["n_outliers"] = 0
        _counts["n_total"] = 0


def measure_bytes_from_tracker(tracker, bits_per_element):
    total_bytes = 0
    for _counts in tracker.values():
        n_total = _counts["n_total"]
        n_outliers = _counts["n_outliers"]
        n_regular = n_total - n_outliers
        total_bytes += math.ceil(n_regular * bits_per_element / 8.0)
        total_bytes += n_outliers * 2  # fp16 value per outlier
        total_bytes += math.ceil(n_total / 8.0)  # 1-bit-per-element bitmap marking which positions are outliers
    return total_bytes


outlier_tracker, _outlier_hook_handles = register_outlier_trackers(model_q)
print(f"Quantization applied to model_q's k_proj/v_proj layers in place; {len(outlier_tracker)} outlier-tracking hooks registered (gated off by default).")
print("model_q: nuq", ABITS, "bit, sparsity_threshold", SPARSITY_THRESHOLD)

## Helper Functions

Shared inference machinery used across all three datasets (WikiText-103,
GSM8K, ARC-Challenge).

In [ ]:
# Block - sync_if_cuda/clear_memory: used across every timed inference
# loop in this notebook (WikiText-103, GSM8K, ARC-Challenge) for
# timing-safe GPU synchronization and between-dataset memory cleanup.


def sync_if_cuda():
    if HAS_CUDA:
        torch.cuda.synchronize()


def clear_memory():
    gc.collect()
    if HAS_CUDA:
        torch.cuda.empty_cache()

In [ ]:
# Block - KV-cache memory measurement (measured from REAL outlier-hook
# data) -- shared inference machinery used by both the WikiText-103 driver
# (via score_chunk_kvquant, WikiText-103 section) and directly by
# ARC-Challenge's gold-choice scoring (ARC-Challenge section). GSM8K
# measures its own memory inline in generate_gsm8k_kvquant instead of
# calling this, but this is still genuine cross-dataset machinery, not
# something specific to any one dataset's driver logic.
#
# Always run as a SEPARATE, UNTIMED pass -- callers run it AFTER their own
# timed loop finishes, so it cannot affect any timing number -- and
# _tracker_enabled is only flipped on for the duration of this one untimed
# forward call, so the outlier-tracking hooks stay inert (immediate no-op)
# everywhere else.
#
# Memory comes from register_outlier_trackers' hooks (model-loading cell,
# Setup): resetting the tracker, running one untimed forward pass over the
# whole chunk with tracking enabled, then reading the REAL per-layer
# outlier counts those hooks captured -- not an assumed fixed sparsity
# fraction.


@torch.no_grad()
def measure_chunk_kv_memory(model_obj, chunk_ids_1d, bits_per_element, tracker=None):
    chunk_ids = chunk_ids_1d.unsqueeze(0).to(device)

    if tracker is not None:
        reset_outlier_tracker(tracker)
        _tracker_enabled[0] = True
        try:
            model_obj(input_ids=chunk_ids, use_cache=False, return_dict=True)
        finally:
            _tracker_enabled[0] = False
        return measure_bytes_from_tracker(tracker, bits_per_element)

    outputs = model_obj(input_ids=chunk_ids, use_cache=True, return_dict=True)
    pkv = outputs.past_key_values
    legacy = pkv.to_legacy_cache() if hasattr(pkv, "to_legacy_cache") else pkv
    total_bytes = 0
    for layer_kv in legacy:
        for t in layer_kv:
            total_bytes += t.numel() * t.element_size()
    return total_bytes

In [ ]:
# Block - Shared multiple-choice (MC) scoring machinery, used by BOTH
# ARC-Challenge and HellaSwag (the two likelihood-based MC datasets in
# this notebook) -- genuinely dataset-agnostic: each dataset's own section
# only builds (prompt, choices, gold_index) and calls
# score_mc_question_kvquant; everything else lives here.
#
# lm_eval_encode_pair(context, choice)
#     Tokenizes context+continuation JOINTLY then splits the result at the
#     context's own token count (matching lm-evaluation-harness) --
#     preserves true tokenizer boundary behavior, which separately
#     tokenizing context and continuation can miss (BPE merges can span
#     the boundary). Left-truncates the context if the combined request
#     would exceed the model's window C.
#
# score_mc_choice_kvquant(model_obj, prompt, choice, bits_per_element, tracker)
#     Times the FULL step-by-step walk over one answer choice's tokens
#     (teacher-forced, exactly like scoring a WikiText-103 chunk) --
#     EVERY choice for a question gets this same real, timed treatment,
#     not just the correct one, since answering a multiple-choice
#     question genuinely requires scoring every candidate (there's no
#     single "real generation" the way GSM8K has). Memory is measured via
#     measure_chunk_kv_memory (previous cell) over the full processed
#     sequence.
#
# score_mc_question_kvquant(model_obj, prompt, choices, gold_index, bits_per_element, tracker)
#     Scores every choice via score_mc_choice_kvquant, then reports BOTH
#     standard MC accuracy metrics: raw (argmin of un-normalized NLL) and
#     normalized (argmin of NLL divided by the choice's own raw character
#     length -- the standard lm-evaluation-harness acc_norm convention).
#     Perplexity comes ONLY from the gold choice's own mean NLL (mirroring
#     GSM8K/ARC's "score only the gold answer" convention for perplexity).
#     TTFT = mean across choices; TBT = weighted mean across every
#     choice's decode steps (not a mean-of-per-choice-TBTs, since choices
#     can have very different lengths); total latency = SUM across
#     choices (the real cost of answering the question is the cost of
#     scoring every choice); peak memory = MAX across choices.


def lm_eval_encode_pair(context, choice):
    context = str(context)
    continuation = " " + str(choice)

    n_spaces = len(context) - len(context.rstrip())
    if n_spaces > 0:
        continuation = context[-n_spaces:] + continuation
        context = context[:-n_spaces]

    if not context:
        raise ValueError("MC context cannot be empty.")

    whole_ids = tokenizer(context + continuation, add_special_tokens=True)["input_ids"]
    context_ids = tokenizer(context, add_special_tokens=True)["input_ids"]
    continuation_ids = whole_ids[len(context_ids):]

    if not context_ids:
        raise ValueError("Context tokenization produced no tokens.")
    if not continuation_ids:
        raise ValueError(f"Continuation tokenization produced no tokens. Context={context!r}, choice={choice!r}")
    if len(continuation_ids) > int(C):
        raise ValueError(f"Choice has {len(continuation_ids)} tokens, exceeding C={C}.")

    max_context_tokens = int(C) + 1 - len(continuation_ids)
    if max_context_tokens < 1:
        raise ValueError("No room remains for a context token.")
    context_ids = context_ids[-max_context_tokens:]

    return context_ids, continuation_ids


@torch.no_grad()
def score_mc_choice_kvquant(model_obj, prompt, choice, bits_per_element, tracker=None):
    context_ids, continuation_ids = lm_eval_encode_pair(prompt, choice)
    full_ids_1d = torch.tensor(context_ids + continuation_ids, device=device)
    n_context = len(context_ids)

    chunk_ids = full_ids_1d.unsqueeze(0)
    input_ids = chunk_ids[:, :-1]
    target_ids = chunk_ids[:, 1:]

    loss_fct = nn.CrossEntropyLoss()
    nll_sum = 0.0
    scored = 0
    step_times = []
    past_key_values = None

    for pos in range(input_ids.shape[1]):
        step_input = input_ids[:, pos:pos + 1]

        sync_if_cuda()
        t0 = time.perf_counter()
        outputs = model_obj(
            input_ids=step_input, past_key_values=past_key_values,
            use_cache=True, return_dict=True,
        )
        sync_if_cuda()
        step_times.append(time.perf_counter() - t0)

        past_key_values = outputs.past_key_values
        step_logits = outputs.logits[:, -1, :]
        step_target = target_ids[:, pos]

        if pos + 1 >= n_context:
            loss = loss_fct(step_logits, step_target)
            nll_sum += loss.float().item()
            scored += 1

    ttft_sec = step_times[0]
    decode_time_sum = sum(step_times[1:])
    decode_steps = len(step_times) - 1
    total_latency_sec = sum(step_times)

    peak_bytes = measure_chunk_kv_memory(model_obj, full_ids_1d, bits_per_element, tracker)

    return {
        "nll_sum": nll_sum, "scored": scored,
        "ttft_sec": ttft_sec, "decode_time_sum": decode_time_sum, "decode_steps": decode_steps,
        "total_latency_sec": total_latency_sec, "peak_memory_bytes": peak_bytes,
        "choice_char_len": max(len(str(choice)), 1),
    }


@torch.no_grad()
def score_mc_question_kvquant(model_obj, prompt, choices, gold_index, bits_per_element, tracker=None):
    choice_results = [
        score_mc_choice_kvquant(model_obj, prompt, choice, bits_per_element, tracker)
        for choice in choices
    ]

    normalized_nlls = [r["nll_sum"] / r["choice_char_len"] for r in choice_results]

    raw_prediction = int(min(range(len(choice_results)), key=lambda i: choice_results[i]["nll_sum"]))
    normalized_prediction = int(min(range(len(choice_results)), key=lambda i: normalized_nlls[i]))

    gold_result = choice_results[gold_index]
    gold_mean_nll = gold_result["nll_sum"] / max(gold_result["scored"], 1)

    total_decode_time = sum(r["decode_time_sum"] for r in choice_results)
    total_decode_steps = sum(r["decode_steps"] for r in choice_results)

    return {
        "raw_prediction": raw_prediction,
        "normalized_prediction": normalized_prediction,
        "raw_correct": int(raw_prediction == gold_index),
        "normalized_correct": int(normalized_prediction == gold_index),
        "perplexity": math.exp(min(gold_mean_nll, 50.0)),
        "ttft_sec": sum(r["ttft_sec"] for r in choice_results) / len(choice_results),
        "tbt_sec": (total_decode_time / total_decode_steps) if total_decode_steps > 0 else 0.0,
        "total_latency_sec": sum(r["total_latency_sec"] for r in choice_results),
        "peak_memory_bytes": max(r["peak_memory_bytes"] for r in choice_results),
    }

## WikiText-103

In [ ]:
# Block - WikiText-103: load the FULL test text, concatenate ALL of
# it into one long sequence, tokenize, split into consecutive chunks of
# length C, then cap to the FIRST N_EVAL_SAMPLES chunks -- a deterministic
# prefix of the real test set, not a random sample. Matches H2O's
# WikiText-103 loading exactly, so the two methods are evaluated on
# identical data. The download goes through robust_call so a transient
# network blip gets retried instead of killing the whole run, clearing its
# HF cache between retries (on_retry), since a broken
# 157MB download can leave a corrupted partial file that plain retries
# just resume (and re-break at the same point) forever.


def chunk_sequence(token_ids_1d, chunk_len):
    chunks = []
    n = token_ids_1d.shape[0]
    for start in range(0, n, chunk_len):
        chunks.append(token_ids_1d[start:start + chunk_len])
    return chunks


def load_wikitext103_chunks():
    testdata = robust_call(
        load_dataset, "wikitext", "wikitext-103-raw-v1", split="test",
        desc="WikiText-103 test load", on_retry=lambda: clear_hf_dataset_cache("wikitext"),
    )
    texts = [t for t in testdata["text"] if len(t.strip()) > 0]
    full_text = "\n\n".join(texts)
    ids = tokenizer(full_text, return_tensors="pt")["input_ids"][0]
    print(f"WikiText-103 test set: {len(texts)} non-empty lines, {ids.shape[0]} tokens total")
    return chunk_sequence(ids, C)[:N_EVAL_SAMPLES]


wikitext103_chunks = load_wikitext103_chunks()
print(f"WikiText-103: {len(wikitext103_chunks)} chunks of up to {C} tokens (first N_EVAL_SAMPLES={N_EVAL_SAMPLES}, not randomly selected)")

In [ ]:
# Block - WikiText-103 step-by-step eval function. TTFT/TBT/perplexity/
# accuracy come ONLY from the timed step-by-step loop below -- the model
# is stepped one token at a time using its own KV cache, teacher-forced
# (real corpus tokens fed in, never the model's own prediction). Memory is
# measured via measure_chunk_kv_memory (Helper Functions, Setup) in a
# SEPARATE, UNTIMED pass run AFTER the timed loop finishes, so it cannot
# affect any timing number.


@torch.no_grad()
def score_chunk_kvquant(model_obj, chunk_ids_1d, bits_per_element, tracker=None):
    L = chunk_ids_1d.shape[0]
    if L < 2:
        return None
    chunk_ids = chunk_ids_1d.unsqueeze(0).to(device)
    input_ids = chunk_ids[:, :-1]
    target_ids = chunk_ids[:, 1:]

    loss_fct = nn.CrossEntropyLoss()
    nll_sum = 0.0
    correct = 0
    scored = 0
    step_times = []
    past_key_values = None

    for pos in range(input_ids.shape[1]):
        step_input = input_ids[:, pos:pos + 1]

        sync_if_cuda()
        t0 = time.perf_counter()
        outputs = model_obj(
            input_ids=step_input, past_key_values=past_key_values,
            use_cache=True, return_dict=True,
        )
        sync_if_cuda()
        step_times.append(time.perf_counter() - t0)

        past_key_values = outputs.past_key_values
        step_logits = outputs.logits[:, -1, :]
        step_target = target_ids[:, pos]

        loss = loss_fct(step_logits, step_target)
        nll_sum += loss.float().item()

        pred = step_logits.argmax(dim=-1)
        correct += int((pred == step_target).sum().item())
        scored += 1

    ttft_sec = step_times[0]
    tbt_sec = sum(step_times[1:]) / len(step_times[1:]) if len(step_times) > 1 else 0.0
    total_latency_sec = sum(step_times)

    peak_bytes = measure_chunk_kv_memory(model_obj, chunk_ids_1d, bits_per_element, tracker)

    return {
        "nll_sum": nll_sum, "scored": scored, "correct": correct,
        "ttft_sec": ttft_sec, "tbt_sec": tbt_sec,
        "total_latency_sec": total_latency_sec, "peak_memory_bytes": peak_bytes,
    }


@torch.no_grad()
def preview_chunk_prediction(model_obj, chunk_ids_1d, n_preview_tokens=30):
    """Untimed, separate forward pass over just the first n_preview_tokens of
    a chunk -- for printing what the model actually predicted vs. the real
    next tokens. Does not affect any reported metric."""
    n_preview_tokens = min(n_preview_tokens, chunk_ids_1d.shape[0] - 1)
    if n_preview_tokens < 1:
        return None
    preview_ids = chunk_ids_1d[:n_preview_tokens + 1].unsqueeze(0).to(device)
    input_ids = preview_ids[:, :-1]
    target_ids = preview_ids[:, 1:]

    outputs = model_obj(input_ids=input_ids, use_cache=False, return_dict=True)
    preds = outputs.logits.argmax(dim=-1)

    return {
        "prompt_text": tokenizer.decode(input_ids[0], skip_special_tokens=True),
        "actual_next_text": tokenizer.decode(target_ids[0], skip_special_tokens=True),
        "predicted_next_text": tokenizer.decode(preds[0], skip_special_tokens=True),
    }

In [ ]:
# Block - WikiText-103 driver: run every chunk through
# score_chunk_kvquant against the 3-bit quantized model.


def evaluate_lm_dataset_kvquant(dataset_name, chunks, model_obj, bits_per_element, method_label, tracker=None):
    clear_memory()
    total_nll = 0.0
    total_scored = 0
    total_correct = 0
    ttft_values, tbt_values, latency_values, peak_mem_values = [], [], [], []

    N_PREVIEW_CHUNKS = 5
    for chunk_idx, chunk_ids in enumerate(tqdm(chunks, desc=f"{dataset_name} | {method_label}")):
        if chunk_idx < N_PREVIEW_CHUNKS:
            preview = preview_chunk_prediction(model_obj, chunk_ids)
            if preview is not None:
                print(f"\n--- {dataset_name} | {method_label} | chunk {chunk_idx} preview ---")
                print(f"Prompt:              {preview['prompt_text']!r}")
                print(f"Actual next text:    {preview['actual_next_text']!r}")
                print(f"Predicted next text: {preview['predicted_next_text']!r}")

        result = score_chunk_kvquant(model_obj, chunk_ids, bits_per_element, tracker)
        if result is None:
            continue
        total_nll += result["nll_sum"]
        total_scored += result["scored"]
        total_correct += result["correct"]
        ttft_values.append(result["ttft_sec"])
        tbt_values.append(result["tbt_sec"])
        latency_values.append(result["total_latency_sec"])
        peak_mem_values.append(result["peak_memory_bytes"])

    avg_nll = total_nll / max(total_scored, 1)
    perplexity = math.exp(min(avg_nll, 50.0))
    accuracy = total_correct / max(total_scored, 1)

    return {
        "dataset": dataset_name,
        "method": method_label,
        "perplexity": perplexity,
        "accuracy": accuracy,
        "ttft_sec": sum(ttft_values) / len(ttft_values) if ttft_values else float("nan"),
        "tbt_sec": sum(tbt_values) / len(tbt_values) if tbt_values else float("nan"),
        "avg_total_latency_sec": sum(latency_values) / len(latency_values) if latency_values else float("nan"),
        "peak_memory_gb": max(peak_mem_values) / 1024**3 if peak_mem_values else 0.0,
        "average_memory_gb": (sum(peak_mem_values) / len(peak_mem_values) / 1024**3) if peak_mem_values else 0.0,
    }


lm_results = []
for _name, _chunks in [("WikiText-103", wikitext103_chunks)]:
    lm_results.append(evaluate_lm_dataset_kvquant(_name, _chunks, model_q, ABITS, METHOD_NAME, tracker=outlier_tracker))

lm_results_df = pd.DataFrame(lm_results)
display(lm_results_df)

## GSM8K

In [ ]:
# Block - GSM8K loading: canonical source, sequential first N_QA_SAMPLES
# questions (no shuffling), few-shot prompt -- identical format to
# H2O_Implementation_v3.ipynb so the two methods' GSM8K prompts and
# question sets can never diverge. Download goes through robust_call,
# clearing its HF cache between retries, so a transient network blip or
# stuck partial download gets retried cleanly instead of killing the whole
# run.


def extract_gsm8k_gold_answer(answer_text):
    m = re.search(r"####\s*(-?[0-9][0-9,]*\.?[0-9]*)", answer_text)
    if not m:
        return None
    try:
        return float(m.group(1).replace(",", ""))
    except ValueError:
        return None


gsm8k_test = robust_call(
    load_dataset, "gsm8k", "main", split="test",
    desc="GSM8K test load", on_retry=lambda: clear_hf_dataset_cache("gsm8k"),
)

gsm8k_qa_pairs = []
for i in range(min(N_QA_SAMPLES, len(gsm8k_test))):
    item = gsm8k_test[i]
    gold = extract_gsm8k_gold_answer(item["answer"])
    if gold is not None:
        gsm8k_qa_pairs.append({"question": item["question"], "gold": gold, "gold_text": item["answer"]})

print(f"GSM8K: {len(gsm8k_qa_pairs)} questions loaded (of {len(gsm8k_test)} total in test set)")

In [ ]:
# Block - GSM8K generation with KVQuant. Uses the real model.generate()
# call -- prefill processes the whole prompt in one shot (building the
# quantized KV cache automatically, since model_q's k_proj/v_proj were
# already swapped for quantizing layers), then decode continues one token
# at a time exactly like a normal generate() loop. A StoppingCriteria
# timestamps every generated token so TTFT/TBT come from this SAME call
# that produces the graded answer -- not a separate probe. Memory is
# measured in a SEPARATE, untimed pass after generate() returns, so it
# cannot affect any timing number -- _tracker_enabled is only flipped on
# for that one untimed pass, so the outlier-tracking hooks stay inert
# during the entire timed generate() call above.


def _extract_final_number(text):
    m = re.search(r"####\s*(-?[0-9][0-9,]*\.?[0-9]*)", text)
    if m:
        num_str = m.group(1)
    else:
        nums = re.findall(r"-?[0-9][0-9,]*\.?[0-9]*", text)
        if not nums:
            return None
        num_str = nums[-1]
    num_str = num_str.replace(",", "").rstrip(".")
    try:
        return float(num_str)
    except ValueError:
        return None


class _TimingCriteria(StoppingCriteria):
    def __init__(self):
        self.token_times = []

    def __call__(self, input_ids, scores, **kwargs):
        self.token_times.append(time.perf_counter())
        return False


@torch.no_grad()
def generate_gsm8k_kvquant(model_obj, question, bits_per_element, tracker=None):
    prompt = GSM8K_FEWSHOT_PREFIX + f"\nQuestion: {question.strip()}\nAnswer:"
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    prompt_len = enc["input_ids"].shape[1]

    timing = _TimingCriteria()
    sync_if_cuda()
    gen_start = time.perf_counter()
    gen_ids = model_obj.generate(
        **enc, max_new_tokens=GSM8K_MAX_NEW_TOKENS, do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        stopping_criteria=StoppingCriteriaList([timing]),
    )
    sync_if_cuda()
    gen_end = time.perf_counter()
    total_latency_sec = gen_end - gen_start

    gen_text = tokenizer.decode(gen_ids[0][prompt_len:], skip_special_tokens=True)
    gen_text = gen_text.split("Question:")[0]

    n_generated = len(timing.token_times)
    if n_generated > 0:
        ttft_sec = timing.token_times[0] - gen_start
    else:
        ttft_sec = total_latency_sec
    tbt_sec = (total_latency_sec - ttft_sec) / max(n_generated - 1, 1)

    total_tokens = prompt_len + n_generated

    # Untimed pass over the full generated sequence, purely to measure
    # memory -- real outlier-hook tracker for the quantized model. Does not
    # affect the timed generate() call above.
    if tracker is not None:
        reset_outlier_tracker(tracker)
        _tracker_enabled[0] = True
        try:
            model_obj(input_ids=gen_ids, use_cache=False, return_dict=True)
        finally:
            _tracker_enabled[0] = False
        peak_bytes = measure_bytes_from_tracker(tracker, bits_per_element)
    else:
        cache_outputs = model_obj(input_ids=gen_ids, use_cache=True, return_dict=True)
        pkv = cache_outputs.past_key_values
        legacy = pkv.to_legacy_cache() if hasattr(pkv, "to_legacy_cache") else pkv
        peak_bytes = sum(t.numel() * t.element_size() for layer_kv in legacy for t in layer_kv)

    return {
        "gen_text": gen_text, "ttft_sec": ttft_sec, "tbt_sec": tbt_sec,
        "total_latency_sec": total_latency_sec, "total_tokens": total_tokens,
        "peak_memory_bytes": peak_bytes,
    }


@torch.no_grad()
def score_gsm8k_perplexity_kvquant(model_obj, question, gold_text):
    """Secondary, untimed teacher-forced pass on the gold answer -- per the
    metrics section: perplexity for GSM8K is usually calculated by teacher-
    forcing the correct answer, not by scoring the generated answer."""
    gold_number = extract_gsm8k_gold_answer(gold_text)
    if gold_number is None:
        return None
    gold_str = str(int(gold_number)) if gold_number == int(gold_number) else str(gold_number)
    prompt = GSM8K_FEWSHOT_PREFIX + f"\nQuestion: {question.strip()}\nAnswer:"
    prompt_ids = tokenizer(prompt, add_special_tokens=True)["input_ids"]
    target_ids = tokenizer(" " + gold_str, add_special_tokens=False)["input_ids"]
    full_ids = torch.tensor([prompt_ids + target_ids], device=device)

    outputs = model_obj(input_ids=full_ids, use_cache=True, return_dict=True)
    logits = outputs.logits[0]
    n_prompt = len(prompt_ids)
    nll_sum = 0.0
    scored = 0
    for ti, target_id in enumerate(target_ids):
        pos = n_prompt - 1 + ti
        if pos < 0 or pos >= logits.shape[0]:
            continue
        log_probs = torch.log_softmax(logits[pos].float(), dim=-1)
        nll_sum += -log_probs[target_id].item()
        scored += 1
    if scored == 0:
        return None
    return math.exp(min(nll_sum / scored, 50.0))

In [ ]:
# Block - GSM8K driver: run every question through generate_gsm8k_kvquant
# (accuracy + TTFT/TBT/latency + memory, all from the SAME real generation
# call plus its untimed memory-measurement pass), plus the secondary
# teacher-forced perplexity pass, against the 3-bit quantized model.


def evaluate_gsm8k_kvquant(qa_pairs, model_obj, bits_per_element, method_label, tracker=None):
    clear_memory()
    correct = 0
    total = 0
    ttft_values, tbt_values, latency_values, peak_mem_values, ppl_values = [], [], [], [], []

    N_PREVIEW_QUESTIONS = 5
    for q_idx, qa in enumerate(tqdm(qa_pairs, desc=f"GSM8K | {method_label}")):
        result = generate_gsm8k_kvquant(model_obj, qa["question"], bits_per_element, tracker)
        pred = _extract_final_number(result["gen_text"])
        is_correct = pred is not None and abs(pred - qa["gold"]) < 1e-4
        correct += int(is_correct)
        total += 1

        if q_idx < N_PREVIEW_QUESTIONS:
            print(f"\n--- GSM8K | {method_label} | question {q_idx} preview ---")
            print(f"Question:    {qa['question']}")
            print(f"Generated:   {result['gen_text'].strip()}")
            print(f"Gold answer: {qa['gold']} | Predicted: {pred} | Correct: {is_correct}")

        ttft_values.append(result["ttft_sec"])
        tbt_values.append(result["tbt_sec"])
        latency_values.append(result["total_latency_sec"])
        peak_mem_values.append(result["peak_memory_bytes"])

        ppl = score_gsm8k_perplexity_kvquant(model_obj, qa["question"], qa["gold_text"])
        if ppl is not None:
            ppl_values.append(ppl)

    accuracy = correct / max(total, 1)
    avg_ppl = sum(ppl_values) / len(ppl_values) if ppl_values else float("nan")

    return {
        "dataset": "GSM8K",
        "method": method_label,
        "perplexity": avg_ppl,
        "accuracy": accuracy,
        "ttft_sec": sum(ttft_values) / len(ttft_values) if ttft_values else float("nan"),
        "tbt_sec": sum(tbt_values) / len(tbt_values) if tbt_values else float("nan"),
        "avg_total_latency_sec": sum(latency_values) / len(latency_values) if latency_values else float("nan"),
        "peak_memory_gb": max(peak_mem_values) / 1024**3 if peak_mem_values else 0.0,
        "average_memory_gb": (sum(peak_mem_values) / len(peak_mem_values) / 1024**3) if peak_mem_values else 0.0,
    }


gsm8k_results = [
    evaluate_gsm8k_kvquant(gsm8k_qa_pairs, model_q, ABITS, METHOD_NAME, tracker=outlier_tracker),
]
gsm8k_results_df = pd.DataFrame(gsm8k_results)
display(gsm8k_results_df)

## ARC-Challenge

Likelihood-based multiple-choice scoring (no generation): every answer
choice is scored (raw and character-length-normalized), perplexity from
the gold choice, TTFT/TBT/latency/memory aggregated across all choices.

In [ ]:
# Block - ARC-Challenge loading: canonical source (allenai/ai2_arc,
# ARC-Challenge config), sequential first N_QA_SAMPLES questions (no
# shuffling), matching the GSM8K loading convention exactly -- first
# min(N_QA_SAMPLES, len(testdata)) items by index, skipping only the
# rare item whose answerKey doesn't match any of its own choice labels
# (a data-quality edge case, not a sampling choice). Download goes through
# robust_call, clearing its HF cache between retries, so a transient
# network blip or stuck partial download gets retried cleanly instead of
# killing the whole run.


def load_arc_challenge_items():
    testdata = robust_call(
        load_dataset, "allenai/ai2_arc", "ARC-Challenge", split="test",
        desc="ARC-Challenge test load", on_retry=lambda: clear_hf_dataset_cache("ai2_arc"),
    )
    items = []
    for i in range(min(N_QA_SAMPLES, len(testdata))):
        row = testdata[i]
        labels = row["choices"]["label"]
        texts = row["choices"]["text"]
        answer_key = row["answerKey"]
        if answer_key not in labels:
            continue
        items.append({
            "question": row["question"],
            "choices": list(zip(labels, texts)),
            "gold_label": answer_key,
        })
    return items


arc_items = load_arc_challenge_items()
print(f"ARC-Challenge: {len(arc_items)} questions loaded (first N_QA_SAMPLES={N_QA_SAMPLES}, not randomly selected)")

In [ ]:
# Block - ARC-Challenge driver: scores every answer choice for each
# question via the shared score_mc_question_kvquant (Helper Functions),
# then reports BOTH standard MC accuracy metrics (raw + character-length
# normalized). Aggregation mirrors GSM8K: perplexity = MEAN of
# per-question perplexities (from each question's gold choice), TTFT/TBT/
# latency = means over questions, peak_memory_gb = max over questions,
# average_memory_gb = mean over questions.


def evaluate_arc_kvquant(items, model_obj, bits_per_element, method_label, tracker=None):
    clear_memory()
    raw_correct = 0
    normalized_correct = 0
    total = 0
    ttft_values, tbt_values, latency_values, peak_mem_values, ppl_values = [], [], [], [], []

    N_PREVIEW_ITEMS = 5
    for idx, item in enumerate(tqdm(items, desc=f"ARC-Challenge | {method_label}")):
        prompt = f"Question: {item['question']}\nAnswer:"
        choice_texts = [text for _, text in item["choices"]]
        gold_index = next(i for i, (label, _) in enumerate(item["choices"]) if label == item["gold_label"])

        result = score_mc_question_kvquant(model_obj, prompt, choice_texts, gold_index, bits_per_element, tracker)

        raw_correct += result["raw_correct"]
        normalized_correct += result["normalized_correct"]
        total += 1

        if idx < N_PREVIEW_ITEMS:
            predicted_label = item["choices"][result["normalized_prediction"]][0]
            print(f"\n--- ARC-Challenge | {method_label} | item {idx} preview ---")
            print(f"Question:   {item['question']}")
            print(f"Choices:    {item['choices']}")
            print(f"Gold label: {item['gold_label']} | Predicted: {predicted_label} | Correct: {bool(result['normalized_correct'])}")

        ttft_values.append(result["ttft_sec"])
        tbt_values.append(result["tbt_sec"])
        latency_values.append(result["total_latency_sec"])
        peak_mem_values.append(result["peak_memory_bytes"])
        ppl_values.append(result["perplexity"])

    accuracy_raw = raw_correct / max(total, 1)
    accuracy_normalized = normalized_correct / max(total, 1)
    avg_ppl = sum(ppl_values) / len(ppl_values) if ppl_values else float("nan")

    return {
        "dataset": "ARC-Challenge",
        "method": method_label,
        "perplexity": avg_ppl,
        "accuracy": accuracy_normalized,
        "accuracy_raw": accuracy_raw,
        "ttft_sec": sum(ttft_values) / len(ttft_values) if ttft_values else float("nan"),
        "tbt_sec": sum(tbt_values) / len(tbt_values) if tbt_values else float("nan"),
        "avg_total_latency_sec": sum(latency_values) / len(latency_values) if latency_values else float("nan"),
        "peak_memory_gb": max(peak_mem_values) / 1024**3 if peak_mem_values else 0.0,
        "average_memory_gb": (sum(peak_mem_values) / len(peak_mem_values) / 1024**3) if peak_mem_values else 0.0,
    }


arc_results = [
    evaluate_arc_kvquant(arc_items, model_q, ABITS, METHOD_NAME, tracker=outlier_tracker),
]
arc_results_df = pd.DataFrame(arc_results)
display(arc_results_df)

## HellaSwag

Likelihood-based multiple-choice scoring (no generation), same
methodology as ARC-Challenge: every answer choice is scored (raw and
character-length-normalized), perplexity from the gold ending,
TTFT/TBT/latency/memory aggregated across all choices.

In [ ]:
# Block - HellaSwag loading: canonical source (Rowan/hellaswag), sequential
# first N_QA_SAMPLES examples (no shuffling) from the validation split --
# HellaSwag's test split has no public labels, so validation is the
# standard evaluation split (matching lm-evaluation-harness convention).
# Preprocessing matches the standard lm-evaluation-harness HellaSwag
# convention: strip "[title]" markers and bracketed annotations, collapse
# double spaces. Download goes through robust_call, clearing its HF cache
# between retries, so a transient network blip or stuck partial download
# gets retried cleanly instead of killing the whole run.


def _hellaswag_preprocess(text):
    """Matches lm-evaluation-harness HellaSwag preprocessing."""
    text = str(text).strip()
    text = text.replace(" [title]", ". ")
    text = re.sub(r"\[.*?\]", "", text)
    text = text.replace("  ", " ")
    return text


def load_hellaswag_items():
    dataset = robust_call(
        load_dataset, "Rowan/hellaswag", split="validation",
        desc="HellaSwag validation load", on_retry=lambda: clear_hf_dataset_cache("hellaswag"),
    )
    items = []
    for i in range(min(N_QA_SAMPLES, len(dataset))):
        row = dataset[i]
        context = str(row["ctx_a"]) + " " + str(row["ctx_b"]).capitalize()
        prompt = _hellaswag_preprocess(str(row["activity_label"]) + ": " + context)
        choices = [_hellaswag_preprocess(choice) for choice in row["endings"]]
        items.append({
            "prompt": prompt,
            "choices": choices,
            "gold_index": int(row["label"]),
        })
    return items


hellaswag_items = load_hellaswag_items()
print(f"HellaSwag: {len(hellaswag_items)} validation examples loaded (first N_QA_SAMPLES={N_QA_SAMPLES}, not randomly selected)")

In [ ]:
# Block - HellaSwag driver: scores every answer choice for each example
# via the shared score_mc_question_kvquant (Helper Functions) -- the same
# machinery ARC-Challenge uses, since both are likelihood-based
# multiple-choice datasets. Aggregation is identical to ARC-Challenge:
# raw + normalized accuracy, perplexity = MEAN of per-question
# perplexities (from each example's gold ending), TTFT/TBT/latency =
# means over examples, peak_memory_gb = max over examples,
# average_memory_gb = mean over examples.


def evaluate_hellaswag_kvquant(items, model_obj, bits_per_element, method_label, tracker=None):
    clear_memory()
    raw_correct = 0
    normalized_correct = 0
    total = 0
    ttft_values, tbt_values, latency_values, peak_mem_values, ppl_values = [], [], [], [], []

    N_PREVIEW_ITEMS = 5
    for idx, item in enumerate(tqdm(items, desc=f"HellaSwag | {method_label}")):
        result = score_mc_question_kvquant(
            model_obj, item["prompt"], item["choices"], item["gold_index"], bits_per_element, tracker,
        )

        raw_correct += result["raw_correct"]
        normalized_correct += result["normalized_correct"]
        total += 1

        if idx < N_PREVIEW_ITEMS:
            print(f"\n--- HellaSwag | {method_label} | item {idx} preview ---")
            print(f"Prompt:     {item['prompt']}")
            print(f"Choices:    {item['choices']}")
            print(f"Gold index: {item['gold_index']} | Predicted: {result['normalized_prediction']} | Correct: {bool(result['normalized_correct'])}")

        ttft_values.append(result["ttft_sec"])
        tbt_values.append(result["tbt_sec"])
        latency_values.append(result["total_latency_sec"])
        peak_mem_values.append(result["peak_memory_bytes"])
        ppl_values.append(result["perplexity"])

    accuracy_raw = raw_correct / max(total, 1)
    accuracy_normalized = normalized_correct / max(total, 1)
    avg_ppl = sum(ppl_values) / len(ppl_values) if ppl_values else float("nan")

    return {
        "dataset": "HellaSwag",
        "method": method_label,
        "perplexity": avg_ppl,
        "accuracy": accuracy_normalized,
        "accuracy_raw": accuracy_raw,
        "ttft_sec": sum(ttft_values) / len(ttft_values) if ttft_values else float("nan"),
        "tbt_sec": sum(tbt_values) / len(tbt_values) if tbt_values else float("nan"),
        "avg_total_latency_sec": sum(latency_values) / len(latency_values) if latency_values else float("nan"),
        "peak_memory_gb": max(peak_mem_values) / 1024**3 if peak_mem_values else 0.0,
        "average_memory_gb": (sum(peak_mem_values) / len(peak_mem_values) / 1024**3) if peak_mem_values else 0.0,
    }


hellaswag_results = [
    evaluate_hellaswag_kvquant(hellaswag_items, model_q, ABITS, METHOD_NAME, tracker=outlier_tracker),
]
hellaswag_results_df = pd.DataFrame(hellaswag_results)
display(hellaswag_results_df)

## Save Results

In [ ]:
# Block - Combine WikiText-103 / GSM8K / ARC-Challenge / HellaSwag results into
# one table with ONLY the specified metrics, then save to CSV. This notebook only ever
# produces "kvquant_3bit" rows -- 2-bit, 4-bit, and the full-precision
# baseline are saved by their own separate notebooks, so there is no
# cross-notebook filename collision risk.
#
# Robust to partial runs: only concatenates whichever of
# lm_results_df / gsm8k_results_df / arc_results_df / hellaswag_results_df
# actually exist in this session, so you can run just a subset of
# datasets' cells without this cell crashing on a NameError for a
# dataframe you never created.

_result_df_names = ["lm_results_df", "gsm8k_results_df", "arc_results_df", "hellaswag_results_df"]
_available_dfs = []
for _name in _result_df_names:
    if _name in globals():
        _available_dfs.append(globals()[_name])
    else:
        print(f"Note: {_name} not found in this session -- skipping (its dataset's cells were not run).")

results_df = pd.concat(_available_dfs, ignore_index=True)
results_df = results_df[[
    "dataset", "method", "perplexity", "accuracy", "accuracy_raw",
    "ttft_sec", "tbt_sec", "avg_total_latency_sec",
    "peak_memory_gb", "average_memory_gb",
]]
display(results_df)

os.makedirs("/content/drive/MyDrive/KVQuant_v3_Results", exist_ok=True)

_path = "/content/drive/MyDrive/KVQuant_v3_Results/kvquant_abits3_results.csv"
results_df.to_csv(_path, index=False)
print(f"Saved to {_path}")